# Monocular 3D Building Reconstruction — 40 Epochs Training on Google Colab GPU

Train **MultiTaskBuildingNet** on Colab T4 GPU for **40 epochs** to achieve paper-level accuracy for:
1. **Building Footprint Segmentation (IoU > 0.80)**
2. **Metric Height Estimation (nDSM Regression, MAE < 3-4m)**
3. **Automatic 3D Mesh Extrusion (LoD1 OBJ)**

---
### Step 1: Verify GPU Acceleration
Ensure your Colab Runtime is set to **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
!nvidia-smi

### Step 2: Unzip the Complete Project & Data Package
Upload `project_colab.zip` (170 MB) directly into the Colab Files panel on the left.

In [ ]:
# Extract full project code, scripts, and preprocessed datasets
!unzip -q project_colab.zip -d ./
!pip install -q -r requirements.txt
!pip install -q mapbox-earcut trimesh shapely

# Verify all directories exist
!ls -la src/data
!ls -la data/bonai
!ls -la data/synrs3d

### Step 3: Stage 1 — BONAI Footprint Pretraining (20 Epochs on GPU)
Pretrains the visual feature encoder and footprint segmentation decoder on BONAI.

In [ ]:
!python scripts/train.py \
  --data-root data/bonai \
  --epochs 20 \
  --batch-size 16 \
  --w-seg 1.0 \
  --w-height 0.0 \
  --checkpoint-name bonai_pretrained.pt

### Step 4: Stage 2 — SynRS3D Height Fine-Tuning (40 Epochs on GPU)
Initializes from the BONAI pretrained weights and fine-tunes both segmentation and metric height estimation for 40 epochs.

In [ ]:
!python scripts/train.py \
  --data-root data/synrs3d \
  --init-weights checkpoints/bonai_pretrained.pt \
  --epochs 40 \
  --batch-size 16 \
  --w-seg 1.0 \
  --w-height 1.0 \
  --checkpoint-name best_model_40epochs.pt

### Step 5: Plot Training Loss & Validation Metrics (For Research Paper / Report)
Generates the loss curves: IoU, F1 score, Height MAE (meters), and Height RMSE across 40 epochs.

In [ ]:
import json
import matplotlib.pyplot as plt

hist_file = "checkpoints/history_best_model_40epochs.json"
with open(hist_file) as f:
    h = json.load(f)

epochs = list(range(1, len(h["train"]) + 1))
train_loss = [x["loss"] for x in h["train"]]
val_loss = [x["loss"] for x in h["val"]]
val_iou = [x["iou"] for x in h["val"]]
val_mae = [x["height_mae_m"] for x in h["val"]]

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, train_loss, label="Train Loss")
plt.plot(epochs, val_loss, label="Val Loss")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(epochs, val_iou, color="green", label="Val IoU")
plt.title("Building Footprint IoU")
plt.xlabel("Epoch")
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(epochs, val_mae, color="crimson", label="Height MAE (m)")
plt.title("Height Estimation Error (MAE)")
plt.xlabel("Epoch")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("training_curves_40epochs.png", dpi=200)
plt.show()
print(f"Best Val IoU: {max(val_iou):.3f} | Lowest Height MAE: {min(val_mae):.2f}m")

### Step 6: Download the Trained Model
Downloads `best_model_40epochs.pt` back to your local computer to use in your local Geo3D Web Dashboard!

In [ ]:
from google.colab import files
files.download('checkpoints/best_model_40epochs.pt')
files.download('training_curves_40epochs.png')